In [1]:
%run T_symb.ipynb
%run helpers.ipynb

import shelve

In [ ]:
class Distr_of_constant_symbol(object):
    def __init__(self,symb,curv):
        """INPUTS:
        * 'symb' - a T_symb object
        * 'curv' - a positive 2-cochain from the cochain complex associated with symb"""
        self.Tanaka_symbol=symb
        self.curv=curv
        self.Jacobi_id_cache={}
        self.basis=[Vector_Field(A.vec,self) for A in symb.basis if A.wght<0]
        self.normalized_diff_ops=[None]*len(self.Tanaka_symbol.basis) # coeff dictionary with keys tuple of ints and values coefficients
        
    def bracket(self,v1,v2):
        """Returns [v1,v2] as a section of self
        INPUTS:
        * 'v1','v2' - Vector_Field objects with self as parent
        """
        g1,g2=[self.Tanaka_symbol.elt(Matrix(v1.vec)),self.Tanaka_symbol.elt(Matrix(v2.vec))]
        r=(g1.ad(g2)).vec
        e1e2=g1.cast_as_ext_elt().wedge(g2.cast_as_ext_elt())
        r+=self.curv.apply_cochain_map(e1e2).vec
        for i in range(len(v1.vec)):
            for j in range(len(v2.vec)):
                r[i]+=(v1.vec[j]*self.abn_ind_der(v2.vec[i],j)-v2.vec[j]*
                       self.abn_ind_der(v1.vec[i],j))
        return Vector_Field(r,self)
    
    def Jacobi_id(self,i,j,k,l):
        # # Maybe I can speed this up by writing it out in terms of curvatures?
        # # It's got to be fast than what I'm doing here.

        # To do: speed this up!
        """Returns the Jacobi indentity in curvatures from cyc_{i,j,k}([[X_i,X_j],X_k]^l)
        """
        if i==j or j==k or k==i: return 0

        if (i,j,k,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(i,j,k,l)]
        if (k,i,j,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(k,i,j,l)]
        if (j,k,i,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(j,k,i,l)]

        ind_list=[i,j,k,i,j,k]
        r=0

        B=self.Tanaka_symbol.basis
        def dwi(p,q,s):
            res1=None
            res0=None
            if q<p: 
                res1=-1
                res0=self.curv.parent.dwi((str(B[q]),str(B[p]),str(B[s])))
            else:
                res1=1
                res0=self.curv.parent.dwi((str(B[p]),str(B[q]),str(B[s])))
            return [res0,res1]
        
        def curv_coeff(p,q,s):
            if p==q: return 0
            result=0
            t,sgn=dwi(p,q,s)
            try: result=result+sgn*self.curv.vd[t[0]][t[1]][t[2]]
            except: pass
            if B[p].wght+B[q].wght-B[s].wght==0:
                result=result+B[p].ad(B[q]).vec[s]
            return result
            # # Missing the weight 0 terms
        
        for n in range(3): # cyclic sum
            a,b,c=ind_list[n:n+3]
            for d in range(3,len(self.basis)+3):
                r=r+curv_coeff(a,b,d)*curv_coeff(d,c,l)
            r=r-self.normal_ind_der(curv_coeff(a,b,l),c)
        return r
    
        vi=Vector_Field([0]*i+[1]+[0]*(len(self.Tanaka_symbol.basis)-i-1),self)
        vj=Vector_Field([0]*j+[1]+[0]*(len(self.Tanaka_symbol.basis)-j-1),self)
        vk=Vector_Field([0]*k+[1]+[0]*(len(self.Tanaka_symbol.basis)-k-1),self)

        r=Vector_Field([0]*len(self.Tanaka_symbol.basis),self)
        ind_list=[vi,vj,vk,vi,vj,vk]

        for n in range(3): # cyclic sum
            a,b,c=ind_list[n:n+3]
            r+=self.bracket(self.bracket(a,b),c)
        i_list=[i,j,k]
        i_list.sort()
        for l1 in range(len(self.Tanaka_symbol.basis)):
            self.Jacobi_id_cache[tuple(i_list+[l1])]=self.normalize_der(r.vec[l1])
        return self.normalize_der(r.vec[l])

    def abn_ind_der(self,ind_expr,i):
        """Returns the derivative of ind_expr in the direction X_i among X_3, X_4,..., X_{2n-1},
        which is a frame on the base manifold.

        Note: This only works for horizontal derivatives!

        INPUTS:
        * 'ind_expr' -- an expression in h, e, y, and indexed objects
        * 'i' -- an integer between 3 and 2n-1
        """
        if type(ind_expr) in [Matrix, ImmutableDenseMatrix, MutableSparseMatrix, ImmutableSparseMatrix]:
            if type(ind_expr)==ImmutableDenseMatrix: result = mut_mat_copy(ind_expr)
            else: result = copy.copy(ind_expr)
            for j in range(shape(result)[0]):
                for k in range(shape(result)[1]):
                    result[j,k]=self.abn_ind_der(result[j,k],i)
            return result
        
        if type(ind_expr)==T_symb_basis_elt: return ind_expr.parent.elt()
        if type(ind_expr)==T_symb_elt:
            v=[self.abn_ind_der(ind_expr.vec[j],i) for j in range(len(ind_expr.vec))]
            return ind_expr.parent.elt(v)
        if isinstance(ind_expr,numbers.Number):
            return 0
        if type(ind_expr)==Add:
            result = Add(*[self.abn_ind_der(A,i) for A in ind_expr.args])
            return result
        if type(ind_expr)==Mul:
            result=0
            for j in range(len(ind_expr.args)):
                result+=self.abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
            return result
        if type(ind_expr)==Pow:
            return ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*self.abn_ind_der(ind_expr.base,i)
        if type(ind_expr)==Indexed:
            base=ind_expr.base
            ind=list(ind_expr.indices)
            return base[ind+[i]]
        if type(ind_expr)==Symbol: return 0
        if type(ind_expr)==exp:
            return ind_expr*self.abn_ind_der(ind_expr.args[0],i)
        
    def normalized_diff_op(self,i):
        if self.normalized_diff_ops[i]==None: self.set_normalized_diff_op(i)
        return self.normalized_diff_ops[i]
        
    def set_normalized_diff_op(self,i):
        """sets the self.normalized_diff_ops[i], which is a dictionary d representing the X_i derivative.
        The keys of d are integer tuples representing iterated fundamental derivatives, and the values
        of d are coefficients."""
        r={}
        b=IndexedBase('beta')

        # Normalize beta[0,0,0,i]
        d=expand(self.normalize_der(self.abn_ind_der(b[0,0,0],i))).as_coefficients_dict()
        for k in d:
            # Distinguish the coeff (not involving b) from the b factors
            coeff=k.as_coeff_mul()[0]*d[k]
            key=None
            for A in k.as_coeff_mul()[1]:
                if type(A)!=Indexed or A.base!=b: coeff=coeff*A
                if type(A)==Indexed and A.base==b: key=A.indices[3:len(A.indices)]
            if key in r: r[key]=r[key]+coeff
            else: r[key]=coeff
        self.normalized_diff_ops[i]=r

    def normal_ind_der(self,expr,i):
        """Returns the derivative of ind_expr in the direction X_i among X_3, X_4,..., X_{2n-1},
        which is a frame on the base manifold, expressed as commutators of those VFs from the -1 piece.

        Note: This only works for horizontal derivatives!

        INPUTS:
        * 'ind_expr' -- an expression in h, e, y, and indexed objects
        * 'i' -- an integer between 3 and 2n-1"""
        
        r=0
        NDO=self.normalized_diff_op(i)
        for k in NDO:
            coeff=NDO[k]
            temp=copy.copy(expr)
            for i in k: temp=self.abn_ind_der(temp,i)
            r+=coeff*temp
        return r

    def are_deriv_normal(self,expr):
        """Returns True if all the directional derivatives of Indexed objects which appear in expr
        are in normal form; that is, they are expressed using only the fundamental directions.
        INPUTS:
        * 'expr' - an algebraic expression in Indexed objects and symbols"""
        if type(expr)==T_symb_elt:
            for a in expr.vec:
                if not self.are_deriv_normal(a): return False
            return True
        if issubclass(type(expr),Tensor_alg_elt):
            for k1 in expr.vd:
                for k2 in expr.vd[k1]:
                    if not self.are_deriv_normal(expr.vd[k1][k2]): return False
            return True
        if type(expr)==Matrix:
            for i in range(shape(expr)[0]):
                for j in range(shape(expr)[1]):
                    if not self.are_deriv_normal(expr[i,j]): return False
            return True
        if type(expr) in [list, tuple]:
            for a in expr:
                if not self.are_deriv_normal(a): return False
            return True
    
        s=Indexed_obj_in_expr(expr)
        for A in s:
            if len(A.indices)>3:
                for i in A.indices[3:len(A.indices)]:
                    if self.Tanaka_symbol.basis[i].wght<-1: return False
        return True
        
    def normalize_der(self,expr):
        """Converts all derivatives to normal form using the bracket relations from self. Normal form means all derivatives
        are in the fundamental directions, directions 3 and 4.

        WARNING: This assumes all IndexedBase objects have 3 indices which are not derivatives (like curvature coefficients)
        INPUTS:
        * 'expr' - an algebraic expression in Indexed objects and symbols
        
        WARNING: This only works for distributions with the symplectic symbol"""

        if self.are_deriv_normal(expr): 
            return expr
        
        if type(expr)==T_symb_elt:
            return expr.parent.elt([self.normalize_der(a) for a in expr.vec])
        if issubclass(type(expr),Tensor_alg_elt):
            new_vd={}
            for k1 in expr.vd:
                new_vd[k1]={}
                for k2 in expr.vd[k1]:
                    new_vd[k1][k2]=self.normalize_der(expr.vd[k1][k2])
            return expr.parent.elt(new_vd)
        if type(expr) in [Matrix,SparseMatrix,ImmutableDenseMatrix,MutableDenseMatrix]:
            r=zeros(*shape(expr))
            for i in range(shape(expr)[0]):
                for j in range(shape(expr)[1]):
                    r[i,j]=self.normalize_der(expr[i,j])
            return r
        if isinstance(expr,numbers.Number): 
            return expr
        if type(expr)==Add:
            return Add(*[self.normalize_der(A) for A in expr.args])
        if type(expr)==Mul:
            return Mul(*[self.normalize_der(A) for A in expr.args])
        if type(expr)==Pow:
            return self.normalize_der(expr.base)**self.normalize_der(expr.exp)
        if type(expr)==Symbol: 
            return expr
        if type(expr)==exp:
            return exp(self.normalize_der(expr.args[0]))
        if type(expr)==Indexed:
            # Find the first abnormal direction k
            b=expr.base
            prefix=list(expr.indices[0:3])
            i=3
            while i<len(expr.indices) and expr.indices[i] in [3,4]:
                prefix.append(expr.indices[i])
                i+=1
            k=expr.indices[len(prefix)]
            postfix=list(expr.indices[len(prefix)+1:len(expr.indices)])

            # [Xi,Xj] = Xk in the Tanaka symbol
            if k in range(5,len(self.Tanaka_symbol.basis)-1):
                i,j=(3,k-1)
            if k==len(self.Tanaka_symbol.basis)-1:
                i,j=(5,len(self.Tanaka_symbol.basis)-3)

            no_post_r=b[prefix+[j,i]]-b[prefix+[i,j]] # Highest order terms, no postfix

            #Kurvs=(Xk-[Xi,Xj]).vec
            Xi=Vector_Field([0]*(i)+[1]+[0]*(len(self.Tanaka_symbol.basis)-i-1),self)
            Xj=Vector_Field([0]*(j)+[1]+[0]*(len(self.Tanaka_symbol.basis)-j-1),self)
            Xk=Vector_Field([0]*(k)+[1]+[0]*(len(self.Tanaka_symbol.basis)-k-1),self)
            Neg_Kurvs=(Xk-self.bracket(Xi,Xj)).vec
            for q in range(len(Neg_Kurvs)):
                no_post_r+=Neg_Kurvs[q]*b[prefix+[q]]

            r=self.normalize_der(no_post_r)
            for a in postfix:
                r=self.normalize_der(self.abn_ind_der(r,a))
            # Before substituting with Jacobi, I should make sure the values of Jacobi 
            # have only normal derivatives
            return r
        print('normalize_der received unhandled type',type(expr))

    def dir_der(self,f,v):
        """Returns the directional derivative of f in the direction v"""
        u=None
        if type(v) in [list,Matrix,SparseMatrix]:
            u=list(v)
        if type(v)==Vector_Field:
            u=list(v.vec)

        r=0
        for i in range(len(u)):
            if u[i]!=0: r+=u[i]*self.abn_ind_der(f,i)
        return r

    def der_term(self,i0,i1,i2,i3):
        """Returns the derivative term for the component of the Bianchi identity
        applied to (i0,i1,i2), which is an element of self.Tanaka_symbol."""
        g=self.Tanaka_symbol
        X0,X1,X2=[g.basis[a] for a in [i0,i1,i2]]
        w0,w1,w2=[None,None,None]
        try: w0=g.ext_alg.elt_from_cd({(str(X1),str(X2)):1})
        except(KeyError): pass
        try: w1=g.ext_alg.elt_from_cd({(str(X2),str(X0)):1})
        except(KeyError): pass
        try: w2=g.ext_alg.elt_from_cd({(str(X0),str(X1)):1})
        except(KeyError): pass

        r=0
        if w0!=None: r+=self.normal_ind_der(self.curv.apply_cochain_map(w0).vec[i3],i0)
        if w1!=None: r+=self.normal_ind_der(self.curv.apply_cochain_map(w1).vec[i3],i1)
        if w2!=None: r+=self.normal_ind_der(self.curv.apply_cochain_map(w2).vec[i3],i2)
        return -r # negative because of differing sign conventions
    
    def cb_term(self,i0,i1,i2,i3):
        """returns the coboundary of K applied to i0,i1,i2 columns of F.
        This is needed because we care about the value as a cochain in C(g,g), not just C(m,g)
        (at least for the purpose of checks)"""
        r=self.Tanaka_symbol.elt()
        X0,X1,X2=[self.Tanaka_symbol.basis[a] for a in [i0,i1,i2]]
    
        for i in range(3):
            Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
            w0=Y0.negative_projection().cast_as_ext_elt().wedge(Y1.negative_projection().cast_as_ext_elt())
            r+=-self.curv.apply_cochain_map(w0).ad(Y2)
            w1=Y0.ad(Y1).negative_projection().cast_as_ext_elt().wedge(Y2.negative_projection().cast_as_ext_elt())
            r+=-self.curv.apply_cochain_map(w1)
        return -r.vec[i3] # negative because of differing sign conventions

    def gerst_term(self,i0,i1,i2,i3):
        X0,X1,X2=[self.Tanaka_symbol.basis[a] for a in [i0,i1,i2]]
        r=self.Tanaka_symbol.elt()
        for i in range(3):
            Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
            Y0=Y0.negative_projection().cast_as_ext_elt()
            Y1=Y1.negative_projection().cast_as_ext_elt()
            Y2=Y2.negative_projection().cast_as_ext_elt()
            w=self.curv.apply_cochain_map(Y0.wedge(Y1)).negative_projection().cast_as_ext_elt()
            r+=self.curv.apply_cochain_map(w.wedge(Y2))
        return r.vec[i3]
    
    def Bianchi(self,i0,i1,i2,i3,subdivide=False):
        """Returns the Bianchi expression corresponding to indices (i,j,k,l) in the frame F. 
        Should be zero modulo the Jacobi identity for the downstairs distribution"""
        r0=self.cb_term(i0,i1,i2,i3)
        r1=self.der_term(i0,i1,i2,i3)
        r2=self.gerst_term(i0,i1,i2,i3)
        if subdivide: return (r0, r1, r2)
        return r0+r1+r2
    
    def Ricci_Id(self,i,j):
        """returns the Ricci identity for self corresponding to indices i and j as a DiffOpDict"""
        if i==j:return DiffOpDict({},self.Tanaka_symbol, self.curv)
        r=DiffOpDict({(i,j):1,(j,i):-1},self.Tanaka_symbol, self.curv)
        vi=Vector_Field(eye(len(self.Tanaka_symbol.basis)).col(i),self)
        vj=Vector_Field(eye(len(self.Tanaka_symbol.basis)).col(j),self)
        w=self.bracket(vi,vj)
        for k in range(len(w.vec)):
            r.d[(k,)]=-w.vec[k]
        r.clear_zeros()
        return r.normal_form()

In [87]:
class Vector_Field(object):
    def __init__(self,v_rep,parent):
        """INPUTS:
        * 'v_rep' - a list or vector representing a distribution in the frame of parent (positive and negative)
        * 'parent' - a Distr_of_constant_symbol object"""
        self.vec=Matrix(v_rep)
        if shape(self.vec)[1]!=1:
            self.vec=self.vec.transpose()
        self.parent=parent

    def bracket(self,other):
        return self.parent.bracket(self,other)
    
    def __str__(self):
        return str(self.vec)

    def __repr__(self):
        return self.vec.__repr__()
    
    def __add__(self,other):
        return Vector_Field(self.vec+other.vec,self.parent)
    
    def __neg__(self):
        return Vector_Field(-self.vec,self.parent)
    
    def __sub__(self,other):
        return self+(-other)

In [ ]:
def Compute_JS_dict(Distr,max_wght,unshelve=False,reshelve=False,name=None):
    """Returns a differntial substitution dictionary representing the
    Jacobi substitutions for the given distribution up to and including max_wght
    
    INPUTS:
    * 'Distr' - A distribution of constant symbol
    * 'max_wght' - a natural number weight
    * 'unshelve' - if True, will use shelved values from file 'Jacobi_ideals'
    * 'reshelve' - if True, will shelve result to 'Jacobi_ideals'
    * 'name' - a title for the distribution; required if unshelving or reshelving"""
    if (unshelve or reshelve) and name==None:
        raise ValueError('Compute_JS_dict: argument name is required when unshelving or reshelving')

    g=Distr.Tanaka_symbol
    result={}
    p=g.basis.index(g.m_basis[0]) # index of first negative element 

    # First, sort the Jacobi indices by weight
    JI_ind_wghts={}
    for i in range(p,len(g.basis)):
        for j in range(i+1,len(g.basis)):
            for k in range(j+1,len(g.basis)):
                for l in range(p,len(g.basis)):
                    w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[l].wght
                    if not w in JI_ind_wghts: JI_ind_wghts[w]=[]
                    JI_ind_wghts[w].append((i,j,k,l))
    
    # If unshelving is available, return that.
    if unshelve:
        with shelve.open('Jacobi_ideals') as shelf:
            try: return shelf[name+'{j}'.format(j=max_wght)]
            except: pass
    
    # if this weight hasn't been shelved, compute it,
    # possibly unshelving along the way
    for i in range(1,max_wght+1):
        time0=time.time()
        unshelved=False
        if unshelve:
            try:
                result=shelf[name+'{j}'.format(j=i)]
                unshelved=True
            except: pass
        if unshelved==False:
            for j in range(len(JI_ind_wghts[i])):
                ind=JI_ind_wghts[i][j]
                process_JI(ind,Distr,result) 
            if reshelve:
                with shelve.open('Jacobi_ideals') as shelf:
                    shelf[name+'{j}'.format(j=i)]=result
            print('Jacobi id of weight',i,'computed in time',hrs_min_sec(time.time()-time0))
    return result

In [1]:
def Process_Ricci_Id(distr,i,j,dds_dict,Jacobi_dict):
    to_add=dds_subs(distr.Ricci_Id(i,j),dds_dict,distr.Tanaka_symbol,-distr.curv)
    to_add=ds_subs(to_add,Jacobi_dict,distr)[0]
    for k in to_add.d:
        to_add.d[k]=simplify(to_add.d[k])
    to_add.clear_zeros()
    add_DiffOp_to_dds_dict(to_add,dds_dict,distr.Tanaka_symbol,-distr.curv)

def dds_back_substitute(d1,d2,symb,curv):
    """Back substitutes d2 into d1
    INPUTS:
    * 'd1,d2' - diff diff subs dictionaries
    * 'symb' - a Tanaka symbol
    * 'curv' - a curvature 2-cochain"""
    # substitute within keys of d1
    for p in d2:
        for k1 in list(d1.keys()):
            if k1 in d1: # the keys of d1 might be modified by add_DiffOp_to_dds_dict
                i=bytes(k1).find(bytes(p))
                if i!=-1:
                    new_op=(DiffOpDict(d1[k1],symb,curv)
                            -DiffOpDict({k1[0:i]:1},symb,curv)*DiffOpDict(d2[p],symb,curv)*DiffOpDict({k1[i+len(p):len(k1)]:1},symb,curv))
                    d1.pop(k1)
                    add_DiffOp_to_dds_dict(new_op,d1,symb,curv)
    
    # Check if more substitutions are needed in the keys
    first_subs_complete=True 
    for p in d2:
        for k1 in d1:
            i=bytes(k1).find(bytes(p))
            if i!=-1: first_subs_complete=False
    if not first_subs_complete: return dds_back_substitute(d1,d2,symb,curv)

    # Sub into the values
    for k in d1:
        d1[k]=dds_subs(d1[k],d2,symb,curv)
    
def dds_subs(expr,dds_dict,symb,curv):
    if type(expr)==cochain:
        r=expr.parent.elt({})
        for a in expr.vd:
            if not a in r.vd: r.vd[a]={}
            for b in expr.vd[a]:
                r.vd[a][b]=dds_subs(expr.vd[a][b],dds_dict,symb,curv)
        return r
    
    if type(expr) in [Matrix,MutableDenseMatrix,ImmutableDenseMatrix]:
        r=Matrix(expr)
        for i in range(shape(r)[0]):
            for j in range(shape(r)[1]):
                r[i,j]=dds_subs(r[i,j],dds_dict,symb,curv)
        return r
    
    if type(expr)==tuple: # this represents a key from a DiffOpDict
        for p in dds_dict:
            i=bytes(expr).find(bytes(p))
            if i!=-1:
                r=DiffOpDict({expr[0:i]:1},symb,curv)*DiffOpDict(dds_dict[p],symb,curv)*DiffOpDict({expr[i+len(p):len(expr)]:1},symb,curv)
                return dds_subs(r,dds_dict,symb,curv)
        return DiffOpDict({expr:1},symb,curv)
    
    if type(expr)==DiffOpDict:
        r=DiffOpDict({},symb,curv)
        for k in expr.d:
            r+=dds_subs(expr.d[k],dds_dict,symb,curv)*dds_subs(k,dds_dict,symb,curv)
        return r
    if not hasattr(expr,'subs'): return expr

    I_set=Indexed_obj_in_expr(expr)
    nd={}
    for I in I_set:
        v=dds_val(dds_dict,I,symb,curv)
        if v!=I: nd[I]=v
    r=expr.xreplace(nd)
    return r

def dds_val(dds_dict,a,symb,curv):
    """returns the value of an indexed object after applying substitutions from dds_dict
    ARGS:
    * 'ind_obj' - an indexed object with base among I, W, K, eta, and alpha
    * 'dds_dict' - a differential differential substution dictionary"""
    basic_len=-1
    if a.base in [IndexedBase('I'),IndexedBase('W')]: basic_len=1
    elif a.base in [IndexedBase('K'),IndexedBase('alpha'),IndexedBase('eta')]: basic_len=3
    else: print('WARNING: dds_subs expects only bases I, W, K, eta, and alpha')
    
    # Pick a key which appears and substitute
    # Careful! Indices are reversed in the operators as compared to the tensors
    k=None
    i=None
    for k1 in dds_dict:
        inds=k1[::-1]
        i1=bytes(a.indices[basic_len:len(a.indices)]).find(bytes(inds))
        if i1!=-1:
            k,i=k1,i1+basic_len
            break
    if i==None: return a
    r=a.base[a.indices[0:i]]
    inds=a.indices[i+len(k):len(a.indices)][::-1]
    postf=DiffOpDict({inds:1},symb,curv)
    curr=DiffOpDict(dds_dict[k],symb,curv)
    r=postf.apply(curr.apply(r))
    # Finally, substitute the rest
    return dds_subs(r,dds_dict,symb,curv)

def add_DiffOp_to_dds_dict(DiffOp,dds_dict,symb,curv):
    """Adds the relation DiffOp=0 to the given dds_dict"""
    DiffOp.clear_zeros()
    DiffOp=DiffOp.normal_form()
    # for k in dds_dict:
    #     for j in dds_dict[k]:
    #         dds_dict[k][j]=dds_dict[k][j] # This is inefficient, but I need to make sure the values are nonzero somehow
    print('Adding DiffOp =',DiffOp,'\n')
    # Pick a key to substitute for
    # Choose the key of greatest length which comes first in revlex ordering
    key_len=-1
    for k in DiffOp.d: 
        if len(k)>key_len: key_len = len(k)
    if key_len==-1: return None
    key_list=sorted([k for k in DiffOp.d if len(k)==key_len])
    key_list.reverse()
    iso_key=None
    for k in key_list:
        if type(simplify(DiffOp.d[k]))==int:
            if sum([symb.basis[i].wght>=0 for i in k])==0:
                iso_key=k
                break
    if iso_key==None: iso_key=key_list[0]
    print('    iso_key =',iso_key)
    DiffOp.d[iso_key]=simplify(DiffOp.d[iso_key])
    if DiffOp.d[iso_key]==0:
        DiffOp.d.pop(iso_key)
        add_DiffOp_to_dds_dict(DiffOp,dds_dict,symb,curv)
        return None
    try: new_val=copy.deepcopy((-Rational(1,DiffOp.d[iso_key])*DiffOp).d)
    except: new_val=copy.deepcopy(((-1/DiffOp.d[iso_key])*DiffOp).d)
    new_val.pop(iso_key)
    for k in new_val:
        new_val[k]=simplify(new_val[k])
    new_dds={iso_key:new_val}
    dds_back_substitute(dds_dict,new_dds,symb,curv)
    dds_dict[iso_key]=new_val